In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


# =========================================================
# 1. 데이터 불러오기
# =========================================================

train = pd.read_csv(
    "/kaggle/input/competitions/titanic/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/titanic/test.csv"
)

y = train["Survived"]


# =========================================================
# 2. 사용할 변수
# =========================================================

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]


# =========================================================
# 3. 데이터 전처리
# =========================================================


#SimpleImputer 결측치(NaN)를 채워주는 클래스
#1) 숫자형
numeric_transformer = SimpleImputer( 
    strategy="median",   #결측치가 있으면 중앙값으로 채워라
    add_indicator=True   #결측치가 있었는지 여부를 알려주는 새로운 컬럼을 추가해라
)

#Pipeline은 여러 작업을 순서대로 수행하는 클래스
#데이터 -> 결측치 채우기 -> OneHot Encoding -> 완료
#2) 범주형
categorical_transformer = Pipeline([
    (
        "imputer", #이 단계의 이름
        SimpleImputer(
            strategy="most_frequent" #최빈값으로 채우기
        )
    ),
    (
        "onehot", #두번째 단계 이름 
        OneHotEncoder( #범주형 데이터를 숫자로 변환
            handle_unknown="ignore" # 학습 때 없던 값이 테스트에서 나와도 에러를 내지 않음 
        )
    )
])

#ColumnTransformer 컬럼마다 다른 전처리를 적용하는 클래스
#3)혼합형

preprocessor = ColumnTransformer([
    (
        "numeric",  #이름
        numeric_transformer,    #위에만든 클래스
        numeric_features    #숫자형 컬럼 목록 EX: Age
    ),
    (
        "categorical",  #이름
        categorical_transformer,    #22
        categorical_features    #범주형 컬럼
    )
])


# =========================================================
# 4. Random Forest 모델
# =========================================================

#입력 -> 전처리 -> Randomforest -> 예측
model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "random_forest",
        RandomForestClassifier( #모델 생성
            n_estimators=500,   #결정트리 개수
            max_depth=6,    
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )
    )
])


# =========================================================
# 5. 전체 train 데이터 학습
# =========================================================

#모델 학습
model.fit(
    train[features],
    y
)


# =========================================================
# 6. test 데이터 예측
# =========================================================

test_predictions = model.predict(
    test[features]
)


# =========================================================
# 7. 제출 파일 생성
# =========================================================

submission = pd.DataFrame({ #DataFrame 생성
    "PassengerId": test["PassengerId"], #PassengerId 복사
    "Survived": test_predictions.astype(int)    #.astype(int)는 예측값을 정수형(0, 1)으로 변환
})

#CSV 파일 저장
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False #0,892,0X -> 892,0
)


# =========================================================
# 8. 결과 확인
# =========================================================

print("파일 크기:", submission.shape)

print("\n열 이름:")
print(submission.columns.tolist())

print("\n결측값:")
print(submission.isna().sum())

print("\n예측 결과 분포:")
print(
    submission["Survived"]
    .value_counts()
    .sort_index()
)

display(submission.head())